In [1]:
from pathlib import Path
import re
import pandas as pd
import pymupdf
from PIL import Image, ImageOps
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\USER\Desktop\tesseract.exe"
)

YEAR = 2026

pdf_path = Path(
    r"C:\Users\USER\Desktop\chirundu_obb_2026.pdf.pdf"
)

output_folder = Path(f"output_{YEAR}")
output_folder.mkdir(exist_ok=True)

assert pdf_path.exists(), "PDF path is incorrect."

print("PDF exists:", pdf_path.exists())
print("Tesseract version:", pytesseract.get_tesseract_version())

PDF exists: True
Tesseract version: 5.5.3.20260724


In [2]:
def ocr_page(pdf_path, page_number, psm=4):
    document = pymupdf.open(pdf_path)

    try:
        page = document[page_number - 1]

        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(300 / 72, 300 / 72),
            alpha=False
        )

        image = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        image = ImageOps.grayscale(image)
        image = image.point(lambda pixel: 0 if pixel < 180 else 255)

        return pytesseract.image_to_string(
            image,
            config=f"--oem 3 --psm {psm}"
        )

    finally:
        document.close()

In [3]:
page_3_text = ocr_page(pdf_path, 3, psm=4)
print(page_3_text)

OUTPUT BASED ANNUAL BUDGET Page 3

HEA 981 CHIRUNDU TOWN COUNCIL

D
01 Local taxes/rates
001 Residential 202,069 202,069 202,069
002 Commercial 547,946 547,946 547,946

001 Personal levy 45,000 45,500 46,000

02 Fees and Charges

002 Survey fees 7,300 7,400 . 7,500
003 Building inspection-fees 20,000 20,500 21,000
004 Plan scrutiny fee 15,000 15,500 16,000
005 Change of premise use 16,800 17,800 18,800
006 Container/Ntemba fees 25,000 25,500 26,000
007 Rentals/lease of Council’s properties 624,000 625,000 626,000
008 Non-Land Application forms fees 100 200 300
011 Search fees 500 600 700
012 Notice board advert fees 35,100 35,100 35,100
013 Market fees 20,000 22,000 . 24,000
014 Parking fees 28,007,320 28,407,320 . 28,807,320
016 Loading fees (buses, trucks, trains, taxies etc.) 39,840 39,940 40,840
020 Hire of halls 10,000 11,000 12,000
033 Refuse disposal 200,000 200,000 200,000
047 Registration of clubs and societies 20,698 20,698 20,698
063 Billboards and banners 40,520 40,520 40,5

In [4]:
document = pymupdf.open(pdf_path)
total_pages = len(document)
document.close()

raw_ocr_rows = []

for page_number in range(1, total_pages + 1):
    print(f"Reading page {page_number} of {total_pages}...")

    raw_ocr_rows.append({
        "year": YEAR,
        "page": page_number,
        "ocr_text": ocr_page(pdf_path, page_number, psm=4),
        "source_file": pdf_path.name
    })

raw_ocr = pd.DataFrame(raw_ocr_rows)

raw_file = output_folder / "chirundu_obb_2026_raw_ocr.csv"

raw_ocr.to_csv(
    raw_file,
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print("OCR complete.")
print("Pages processed:", len(raw_ocr))
print("Raw file created:", raw_file)

Reading page 1 of 57...
Reading page 2 of 57...
Reading page 3 of 57...
Reading page 4 of 57...
Reading page 5 of 57...
Reading page 6 of 57...
Reading page 7 of 57...
Reading page 8 of 57...
Reading page 9 of 57...
Reading page 10 of 57...
Reading page 11 of 57...
Reading page 12 of 57...
Reading page 13 of 57...
Reading page 14 of 57...
Reading page 15 of 57...
Reading page 16 of 57...
Reading page 17 of 57...
Reading page 18 of 57...
Reading page 19 of 57...
Reading page 20 of 57...
Reading page 21 of 57...
Reading page 22 of 57...
Reading page 23 of 57...
Reading page 24 of 57...
Reading page 25 of 57...
Reading page 26 of 57...
Reading page 27 of 57...
Reading page 28 of 57...
Reading page 29 of 57...
Reading page 30 of 57...
Reading page 31 of 57...
Reading page 32 of 57...
Reading page 33 of 57...
Reading page 34 of 57...
Reading page 35 of 57...
Reading page 36 of 57...
Reading page 37 of 57...
Reading page 38 of 57...
Reading page 39 of 57...
Reading page 40 of 57...
Reading p

In [5]:
extracted_columns = [
    "year",
    "category",
    "programme_or_output",
    "page",
    "budget_amount",
    "currency",
    "notes",
    "source_file",
    "target_value",
    "target_unit"
]

extracted_rows = []
financial_rows = []

for page_number in [3, 4]:
    page_text = raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ].iloc[0]

    current_group = None

    for line in page_text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        group_match = re.match(r"^(0[1-9])\s+([A-Za-z].+)$", line)

        if group_match:
            current_group = group_match.group(2)
            continue

        item_match = re.match(
            r"^(\d{3})\s+(.+?)\s+(\d[\d,]*)\s+(\d[\d,]*)\s+(\d[\d,]*)$",
            line
        )

        if not item_match or current_group is None:
            continue

        revenue_code = item_match.group(1)
        item_name = item_match.group(2).strip()

        # First number is the 2026 budget.
        budget_2026 = int(item_match.group(3).replace(",", ""))

        if "Constituency Development Fund" in item_name:
            category = "CDF"
        elif "Local Government Equalisation Fund" in item_name:
            category = "LGEF"
        elif current_group in [
            "Local taxes/rates",
            "Fees and Charges",
            "Licenses",
            "Levies",
            "Permits",
            "Charges",
            "Other Incomes"
        ]:
            category = "Local Revenue"
        else:
            category = "Revenue"

        financial_rows.append({
            "year": YEAR,
            "category": category,
            "programme_or_output": item_name,
            "page": page_number,
            "budget_amount": budget_2026,
            "currency": "ZMW",
            "notes": f"Revenue code {revenue_code}; group: {current_group}.",
            "source_file": pdf_path.name,
            "target_value": None,
            "target_unit": None
        })

financial_df = pd.DataFrame(
    financial_rows,
    columns=extracted_columns
)

display(financial_df)
print("Financial rows extracted:", len(financial_df))

,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2026,Local Revenue,Residential,3,202069,ZMW,Revenue code 001; group: Local taxes/rates.,chirundu_obb_2026.pdf.pdf,None,None
1,2026,Local Revenue,Commercial,3,547946,ZMW,Revenue code 002; group: Local taxes/rates.,chirundu_obb_2026.pdf.pdf,None,None
2,2026,Local Revenue,Personal levy,3,45000,ZMW,Revenue code 001; group: Local taxes/rates.,chirundu_obb_2026.pdf.pdf,None,None
3,2026,Local Revenue,Building inspection-fees,3,20000,ZMW,Revenue code 003; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
4,2026,Local Revenue,Plan scrutiny fee,3,15000,ZMW,Revenue code 004; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
5,2026,Local Revenue,Change of premise use,3,16800,ZMW,Revenue code 005; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
6,2026,Local Revenue,Container/Ntemba fees,3,25000,ZMW,Revenue code 006; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
7,2026,Local Revenue,Rentals/lease of Council’s properties,3,624000,ZMW,Revenue code 007; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
8,2026,Local Revenue,Non-Land Application forms fees,3,100,ZMW,Revenue code 008; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
9,2026,Local Revenue,Search fees,3,500,ZMW,Revenue code 011; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None


Financial rows extracted: 47


In [6]:
extracted_columns = [
    "year",
    "category",
    "programme_or_output",
    "page",
    "budget_amount",
    "currency",
    "notes",
    "source_file",
    "target_value",
    "target_unit"
]

financial_rows = []

for page_number in [3, 4]:
    page_text = raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ].iloc[0]

    current_group = None

    for line in page_text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        # Remove OCR dots inserted between figures.
        line = re.sub(r"\s+\.\s+", " ", line)

        group_match = re.match(r"^(0[1-9])\s+([A-Za-z].+)$", line)

        if group_match:
            current_group = group_match.group(2)
            continue

        item_match = re.match(
            r"^(\d{3})\s+(.+?)\s+(\d[\d,]*)\s+(\d[\d,]*)\s+(\d[\d,]*)$",
            line
        )

        if not item_match or current_group is None:
            continue

        revenue_code = item_match.group(1)
        item_name = item_match.group(2).strip()
        budget_2026 = int(item_match.group(3).replace(",", ""))

        if "Constituency Development Fund" in item_name:
            category = "CDF"
        elif "Local Government Equalisation Fund" in item_name:
            category = "LGEF"
        elif current_group in [
            "Local taxes/rates",
            "Fees and Charges",
            "Licenses",
            "Levies",
            "Permits",
            "Charges",
            "Other Incomes"
        ]:
            category = "Local Revenue"
        else:
            category = "Revenue"

        financial_rows.append({
            "year": YEAR,
            "category": category,
            "programme_or_output": item_name,
            "page": page_number,
            "budget_amount": budget_2026,
            "currency": "ZMW",
            "notes": f"Revenue code {revenue_code}; group: {current_group}.",
            "source_file": pdf_path.name,
            "target_value": None,
            "target_unit": None
        })

financial_df = pd.DataFrame(
    financial_rows,
    columns=extracted_columns
)

print(financial_df["category"].value_counts())
display(financial_df)

category
Local Revenue    49
Revenue           4
Name: count, dtype: int64


,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2026,Local Revenue,Residential,3,202069,ZMW,Revenue code 001; group: Local taxes/rates.,chirundu_obb_2026.pdf.pdf,None,None
1,2026,Local Revenue,Commercial,3,547946,ZMW,Revenue code 002; group: Local taxes/rates.,chirundu_obb_2026.pdf.pdf,None,None
2,2026,Local Revenue,Personal levy,3,45000,ZMW,Revenue code 001; group: Local taxes/rates.,chirundu_obb_2026.pdf.pdf,None,None
3,2026,Local Revenue,Survey fees,3,7300,ZMW,Revenue code 002; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
4,2026,Local Revenue,Building inspection-fees,3,20000,ZMW,Revenue code 003; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
5,2026,Local Revenue,Plan scrutiny fee,3,15000,ZMW,Revenue code 004; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
6,2026,Local Revenue,Change of premise use,3,16800,ZMW,Revenue code 005; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
7,2026,Local Revenue,Container/Ntemba fees,3,25000,ZMW,Revenue code 006; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
8,2026,Local Revenue,Rentals/lease of Council’s properties,3,624000,ZMW,Revenue code 007; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None
9,2026,Local Revenue,Non-Land Application forms fees,3,100,ZMW,Revenue code 008; group: Fees and Charges.,chirundu_obb_2026.pdf.pdf,None,None


In [7]:
print(
    raw_ocr.loc[
        raw_ocr["page"] == 4,
        "ocr_text"
    ].iloc[0]
)

Page 4 OUTPUT BASED ANNUAL BUDGET

HEA 981 CHIRUNDU TOWN COUNCIL

D

04 Levies

001 Livestock Movement levy 555,000 600,000 750,000
003 Fish levy 10,000 10,500 11,000
004 Pole levy 1,500 1,700 1,900
006 Sand levy 30,739 32,739 34,739
010 Timber Levy 50,000 55,000 . 60,000
021 Manufacturing 20,615 22,615 24,615

05 Permits

001 Health permits 795,900 795,900 795,900
003 Herbalist permit 2,500 3,000 3,500
005 Transportation of meat products 2,500 2,600 2,800
006 Transportation of opaque beer 113,600 115,600 117,600
008 Burial permits and grave sites 20,000 22,000 24,000
009 Fire certificate 600,000 600,600 600,900
010 Extension of Business hours permits 2,752,000 2,852,000 2,952,000
011 Social gathering permit 3,750 4,750 . 4,750
099 Primary, Secondary and Tertiary permits 23,750 21,750 . 22,750

06 Charges

001 Service Charges Residential plots 10,450 0 0
002 Service Charges Industrial plots 25,000 0 0

004 Premium Plot Commercial 1,000,000 0 0

07 Other Incomes
002 Surplus/ Deficit fro

In [ ]:
financial_df.loc[len(financial_df)] = {
    "year": YEAR,
    "category": "CDF",
    "programme_or_output": "Constituency Development Fund",
    "page": 4,
    "budget_amount": CDF_AMOUNT_FROM_PDF,
    "currency": "ZMW",
    "notes": "Revenue code 08-001; added after OCR review.",
    "source_file": pdf_path.name,
    "target_value": None,
    "target_unit": None
}

financial_df.loc[len(financial_df)] = {
    "year": YEAR,
    "category": "LGEF",
    "programme_or_output": "Local Government Equalisation Fund",
    "page": 4,
    "budget_amount": LGEF_AMOUNT_FROM_PDF,
    "currency": "ZMW",
    "notes": "Revenue code 08-004; added after OCR review.",
    "source_file": pdf_path.name,
    "target_value": None,
    "target_unit": None
}

In [9]:
page_4_text = ocr_page(pdf_path, 4, psm=6)
print(page_4_text)

Page 4 OUTPUT BASED ANNUAL BUDGET
HEA 981 CHIRUNDU TOWN COUNCIL
D
04 Levies
001 Livestock Movement levy 555,000 600,000 750,000
003 Fish levy 10,000 10,500 11,000
004 Pole levy 1,500 1,700 1,900
006 Sand levy 30,739 32,739 34,739
a
001 Health permits 795,900 795,900 795,900
003 Herbalist permit 2,500 3,000 3,500
005 Transportation of meat products 2,500 2,600 2,800
| 010 Extension of Business hours permits 2,752,000 | 2,852,000 . 2,952,000
| 011 Social gathering permit | 3,750 | 4,750 . 4,750
099 Primary, Secondary and Tertiary permits 23,750 21,750 22,750
CM ches
001 Service Charges Residential plots 10,450 0 0
07 Other Incomes
002 Surplus/ Deficit from Commercial Ventures 2,200,400 2,300,400 . 2,500,400
002 Roads Grant 6,351,370 6,351,370 6,351,370
Ti onorsupport (Grams)
001 Devolution Capital Grant 11,554,159 11,554,159 11,554,159



In [10]:
page_4_sparse = ocr_page(pdf_path, 4, psm=11)
print(page_4_sparse)

Page 4

OUTPUT BASED ANNUAL BUDGET

HEA 981 CHIRUNDU TOWN COUNCIL

D

ee

04

Levies

001

Livestock Movement levy

555,000

600,000

750,000

003

Fish levy

10,000

10,500

11,000

004

Pole levy

1,500

1,700

1,900

006

Sand levy

30,739

32,739

34,739

010

Timber Levy

50,000

55,000

60,000

021

Manufacturing

20,615

22,615

24,615

05

Permits

001

Health permits

795,900

795,900

795,900

003

Herbalist permit

2,500

3,000

3,500

005

Transportation of meat products

2,500

2,600

2,800

006

Transportation of opaque beer

113,600

115,600

117,600

008

20,000

22,000

24,000

Burial permits and grave sites

009

Fire certificate

600,000

600,600

600,900

010

Extension of Business hours permits

2,752,000

2,852,000

2,952,000

011

Social gathering permit

3,750

4,750

4,750

099

Primary, Secondary and Tertiary permits

23,750

21,750

22,750

06

Charges

001

Service Charges Residential plots

10,450

002

Service Charges Industrial plots

25,000

004

Premium

In [11]:
page_5_sparse = ocr_page(pdf_path, 5, psm=11)
print(page_5_sparse)

OUTPUT BASED ANNUAL BUDGET

Page 5

HEA 981 CHIRUNDU TOWN COUNCIL

D

ee

10

Local Development Fund

001

Constituency Development Fund

40,032,550

40,032,550

40,032,550

002

LGEF

8,400,000

8,400,000

8,400,000

003

Grant in Lieu of Rates

150,000

150,000

150,000

OO —SCSCs

ee

4.0 BUDGET SUMMARY

The Chirundu Town Council Budget stands at K112.4 million, an increase of 8 percent when compared to

the 2025 budget allocation that stood at K103.9 million.

The increase is mainly attributed to a 11 percent increment in the 2026 Constituency Development Fund

(CDF) allocation of K40 million, with a K11.5 million provision for Capital Grant funded by World Bank,

including a 14 percent increase in the projected locally mobilised revenue of K39.4 million.

The Local Authority will also receive a K6.3 million Grant from vehicle licencing and a K2.9 million

provision for Cash For Work (CFW). The budget has been allocated to sixteen (16) programmes planned

for implementation.

Table

In [12]:
financial_df.loc[len(financial_df)] = {
    "year": YEAR,
    "category": "CDF",
    "programme_or_output": "Constituency Development Fund",
    "page": 5,
    "budget_amount": 40032550,
    "currency": "ZMW",
    "notes": "Revenue code 001; group: Local Development Fund.",
    "source_file": pdf_path.name,
    "target_value": None,
    "target_unit": None
}

financial_df.loc[len(financial_df)] = {
    "year": YEAR,
    "category": "LGEF",
    "programme_or_output": "Local Government Equalisation Fund",
    "page": 5,
    "budget_amount": 8400000,
    "currency": "ZMW",
    "notes": "Revenue code 002; group: Local Development Fund.",
    "source_file": pdf_path.name,
    "target_value": None,
    "target_unit": None
}

financial_df.loc[len(financial_df)] = {
    "year": YEAR,
    "category": "Revenue",
    "programme_or_output": "Grant in Lieu of Rates",
    "page": 5,
    "budget_amount": 150000,
    "currency": "ZMW",
    "notes": "Revenue code 003; group: Local Development Fund.",
    "source_file": pdf_path.name,
    "target_value": None,
    "target_unit": None
}

print(financial_df["category"].value_counts())
print("Financial rows:", len(financial_df))

category
Local Revenue    49
Revenue           5
CDF               1
LGEF              1
Name: count, dtype: int64
Financial rows: 56


In [13]:
print(
    raw_ocr.loc[
        raw_ocr["page"] == 7,
        "ocr_text"
    ].iloc[0]
)

OUTPUT BASED ANNUAL BUDGET Page7

HEA 981 CHIRUNDU TOWN COUNCIL
D

Table:2 Budget Allocation by Programme

2024 2025 2026
Code Programme Approved Approved Budget

Budget(K) Budget(K) | Estimates(K)

1 Constituency Development 30,635,642 36,058,150 40,032,550
2 Local Governance 1,377,628 3,548,140 3,622,278
3 Integrated Development Planning 3,095,245 5,329,132 4,222,250
4 Economic and Business Development 676,430 1,007,948 776,800
5 Public Health and Environmental Protection 3,715,585 4,038,834 3,962,957
6 Housing and Community Amenities 8,444,722 10,232,353 15,857,096
7 Recreation Culture and Religion 642,134 736,244 642,904
8 Education and Skills Development 228,100 19,241 19,241
10 Public Order and Safety 677,220 5,253,277 5,502,186
11 Management and Support Services 10,951,590 13,868,479 18,783,480
12 Resource Mobilisation and Management 9,150,433 6,883,118 4,879,538
13 District Health servcies 1,457,388 1,467,387 1,540,765
15 Transport Services 3,742,847 3,252,077 6,402,860
16 Agri

In [14]:
programme_rows = []

page_7_text = raw_ocr.loc[
    raw_ocr["page"] == 7,
    "ocr_text"
].iloc[0]

for line in page_7_text.splitlines():
    line = re.sub(r"\s+", " ", line).strip()

    programme_match = re.match(
        r"^(\d{1,2})\s+(.+?)\s+"
        r"(?:\(0\)|-|\d[\d,]*)\s+"
        r"(?:\(0\)|-|\d[\d,]*)\s+"
        r"(\d[\d,]*)$",
        line
    )

    if programme_match:
        programme_code = programme_match.group(1)
        programme_name = programme_match.group(2).strip()
        budget_2026 = int(programme_match.group(3).replace(",", ""))

        programme_rows.append({
            "year": YEAR,
            "category": "Programme",
            "programme_or_output": programme_name,
            "page": 7,
            "budget_amount": budget_2026,
            "currency": "ZMW",
            "notes": f"Programme code {programme_code}; 2026 budget allocation.",
            "source_file": pdf_path.name,
            "target_value": None,
            "target_unit": None
        })

programmes_df = pd.DataFrame(
    programme_rows,
    columns=extracted_columns
)

programmes_df["programme_or_output"] = programmes_df[
    "programme_or_output"
].replace(
    "District Health servcies",
    "District Health Services"
)

display(programmes_df)
print("Programme rows found:", len(programmes_df))

,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit
0,2026,Programme,Constituency Development,7,40032550,ZMW,Programme code 1; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
1,2026,Programme,Local Governance,7,3622278,ZMW,Programme code 2; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
2,2026,Programme,Integrated Development Planning,7,4222250,ZMW,Programme code 3; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
3,2026,Programme,Economic and Business Development,7,776800,ZMW,Programme code 4; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
4,2026,Programme,Public Health and Environmental Protection,7,3962957,ZMW,Programme code 5; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
5,2026,Programme,Housing and Community Amenities,7,15857096,ZMW,Programme code 6; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
6,2026,Programme,Recreation Culture and Religion,7,642904,ZMW,Programme code 7; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
7,2026,Programme,Education and Skills Development,7,19241,ZMW,Programme code 8; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
8,2026,Programme,Public Order and Safety,7,5502186,ZMW,Programme code 10; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None
9,2026,Programme,Management and Support Services,7,18783480,ZMW,Programme code 11; 2026 budget allocation.,chirundu_obb_2026.pdf.pdf,None,None


Programme rows found: 16


In [15]:
for page_number in range(12, total_pages + 1):
    print(f"Re-reading page {page_number} of {total_pages}...")

    raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ] = ocr_page(pdf_path, page_number, psm=6)

raw_ocr.to_csv(
    output_folder / "chirundu_obb_2026_raw_ocr.csv",
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print("Output-page OCR complete.")

Re-reading page 12 of 57...
Re-reading page 13 of 57...
Re-reading page 14 of 57...
Re-reading page 15 of 57...
Re-reading page 16 of 57...
Re-reading page 17 of 57...
Re-reading page 18 of 57...
Re-reading page 19 of 57...
Re-reading page 20 of 57...
Re-reading page 21 of 57...
Re-reading page 22 of 57...
Re-reading page 23 of 57...
Re-reading page 24 of 57...
Re-reading page 25 of 57...
Re-reading page 26 of 57...
Re-reading page 27 of 57...
Re-reading page 28 of 57...
Re-reading page 29 of 57...
Re-reading page 30 of 57...
Re-reading page 31 of 57...
Re-reading page 32 of 57...
Re-reading page 33 of 57...
Re-reading page 34 of 57...
Re-reading page 35 of 57...
Re-reading page 36 of 57...
Re-reading page 37 of 57...
Re-reading page 38 of 57...
Re-reading page 39 of 57...
Re-reading page 40 of 57...
Re-reading page 41 of 57...
Re-reading page 42 of 57...
Re-reading page 43 of 57...
Re-reading page 44 of 57...
Re-reading page 45 of 57...
Re-reading page 46 of 57...
Re-reading page 47 o

In [16]:
output_rows = []

for _, row in raw_ocr.iterrows():
    page_text = row["ocr_text"]

    programme_match = re.search(
        r"Programme\s*:?\s*0*(\d+)\s*:?\s*([A-Za-z][^\n]+)",
        page_text,
        flags=re.IGNORECASE
    )

    output_section = re.search(
        r"Table\s*6\s*:\s*Programme Outputs(.*?)(?:Executive Authority|Controlling Officer)",
        page_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not programme_match or not output_section:
        continue

    programme_number = programme_match.group(1)
    programme_name = programme_match.group(2).strip()
    current_key_output = None

    for line in output_section.group(1).splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        if not line or "Key Output" in line or "Target" in line or "Actual" in line:
            continue

        indicator_match = re.match(r"^(\d{2})\.?\s+(.+)$", line)

        if indicator_match:
            indicator_number = indicator_match.group(1)
            indicator_and_values = indicator_match.group(2)

            values = re.findall(
                r"(?<![A-Za-z])(?:\(?\d[\d,]*\)?|-)(?![A-Za-z])",
                indicator_and_values
            )

            target_raw = values[-1] if values else None

            indicator = re.sub(
                r"\s+(?:\(?\d[\d,]*\)?|-)"
                r"(?:\s+(?:\(?\d[\d,]*\)?|-))*\s*$",
                "",
                indicator_and_values
            ).strip()

            if "percentage" in indicator.lower():
                target_unit = "percent"
            elif "number" in indicator.lower():
                target_unit = "count"
            else:
                target_unit = "value"

            if target_raw in [None, "-"]:
                target_value = None
            else:
                target_value = int(
                    target_raw
                    .replace("(", "")
                    .replace(")", "")
                    .replace(",", "")
                )

            output_rows.append({
                "year": YEAR,
                "category": "Output",
                "programme_or_output": indicator,
                "page": row["page"],
                "budget_amount": None,
                "currency": "ZMW",
                "notes": (
                    f"Programme {programme_number}: {programme_name}; "
                    f"Key output: {current_key_output}; "
                    f"Indicator {indicator_number}."
                ),
                "source_file": pdf_path.name,
                "target_value": target_value,
                "target_unit": target_unit
            })

outputs_df = pd.DataFrame(
    output_rows,
    columns=extracted_columns
)

outputs_df["programme_code"] = outputs_df["notes"].str.extract(
    r"Programme\s+(\d+)"
)[0]

print("Output rows found:", len(outputs_df))
display(outputs_df.head(10))

Output rows found: 122


,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit,programme_code
0,2026,Output,Kilometres of Roads graded,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,55,value,1
1,2026,Output,Number of Maternity Annexes constructed,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,1,count,1
2,2026,Output,Percentage of approved grant applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1
3,2026,Output,Percentage of approved loan applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1
4,2026,Output,Number of CDF meetings held,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,6,count,1
5,2026,Output,Number of Field appraisal reports produced,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,2,count,1
6,2026,Output,Number of Monitoring reports produced,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,5,count,1
7,2026,Output,Number of Desk Appraisal Reports produced,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,2,count,1
8,2026,Output,Percentage approved Skills bursaries applicant...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1
9,2026,Output,Percentage approved Secondary bursaries applic...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1


In [17]:
expected_programmes = set(
    programmes_df["notes"].str.extract(
        r"Programme code (\d+)"
    )[0].dropna()
)

found_programmes = set(outputs_df["programme_code"].dropna())

print("Expected programmes:", sorted(expected_programmes, key=int))
print("Missing output programmes:", sorted(
    expected_programmes - found_programmes,
    key=int
))

Expected programmes: ['1', '2', '3', '4', '5', '6', '7', '8', '10', '11', '12', '13', '15', '16', '17', '18']
Missing output programmes: ['16', '17']


In [18]:
for page_number in range(12, total_pages + 1):
    print(f"Re-reading output page {page_number} of {total_pages}...")

    raw_ocr.loc[
        raw_ocr["page"] == page_number,
        "ocr_text"
    ] = ocr_page(pdf_path, page_number, psm=6)

raw_ocr.to_csv(
    output_folder / "chirundu_obb_2026_raw_ocr.csv",
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print("Output-page OCR complete.")

Re-reading output page 12 of 57...
Re-reading output page 13 of 57...
Re-reading output page 14 of 57...
Re-reading output page 15 of 57...
Re-reading output page 16 of 57...
Re-reading output page 17 of 57...
Re-reading output page 18 of 57...
Re-reading output page 19 of 57...
Re-reading output page 20 of 57...
Re-reading output page 21 of 57...
Re-reading output page 22 of 57...
Re-reading output page 23 of 57...
Re-reading output page 24 of 57...
Re-reading output page 25 of 57...
Re-reading output page 26 of 57...
Re-reading output page 27 of 57...
Re-reading output page 28 of 57...
Re-reading output page 29 of 57...
Re-reading output page 30 of 57...
Re-reading output page 31 of 57...
Re-reading output page 32 of 57...
Re-reading output page 33 of 57...
Re-reading output page 34 of 57...
Re-reading output page 35 of 57...
Re-reading output page 36 of 57...
Re-reading output page 37 of 57...
Re-reading output page 38 of 57...
Re-reading output page 39 of 57...
Re-reading output pa

In [19]:
output_rows = []

for _, row in raw_ocr.iterrows():
    page_text = row["ocr_text"]

    programme_match = re.search(
        r"Programme\s*:?\s*0*(\d+)\s*:?\s*([A-Za-z][^\n]+)",
        page_text,
        flags=re.IGNORECASE
    )

    output_section = re.search(
        r"Table\s*6\s*:\s*Programme Outputs(.*?)(?:Executive Authority|Controlling Officer)",
        page_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not programme_match or not output_section:
        continue

    programme_code = programme_match.group(1)
    programme_name = programme_match.group(2).strip()
    key_output = None

    for line in output_section.group(1).splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        if not line or "Key Output" in line or "Target" in line or "Actual" in line:
            continue

        indicator_match = re.match(r"^(\d{2})\.?\s+(.+)$", line)

        if not indicator_match:
            key_output = line
            continue

        indicator_number = indicator_match.group(1)
        text_and_values = indicator_match.group(2)

        values = re.findall(
            r"(?<![A-Za-z])(?:\(?\d[\d,]*\)?|-)(?![A-Za-z])",
            text_and_values
        )

        target_raw = values[-1] if values else None

        indicator = re.sub(
            r"\s+(?:\(?\d[\d,]*\)?|-)"
            r"(?:\s+(?:\(?\d[\d,]*\)?|-))*\s*$",
            "",
            text_and_values
        ).strip()

        if target_raw in [None, "-"]:
            target_value = None
        else:
            target_value = int(
                target_raw.replace("(", "").replace(")", "").replace(",", "")
            )

        if "percentage" in indicator.lower():
            target_unit = "percent"
        elif "number" in indicator.lower():
            target_unit = "count"
        else:
            target_unit = "value"

        output_rows.append({
            "year": YEAR,
            "category": "Output",
            "programme_or_output": indicator,
            "page": row["page"],
            "budget_amount": None,
            "currency": "ZMW",
            "notes": (
                f"Programme {programme_code}: {programme_name}; "
                f"Key output: {key_output}; "
                f"Indicator {indicator_number}."
            ),
            "source_file": pdf_path.name,
            "target_value": target_value,
            "target_unit": target_unit
        })

outputs_df = pd.DataFrame(output_rows, columns=extracted_columns)

outputs_df["programme_code"] = outputs_df["notes"].str.extract(
    r"Programme\s+(\d+)"
)[0]

print("Output rows found:", len(outputs_df))
display(outputs_df.head(10))

Output rows found: 122


,year,category,programme_or_output,page,budget_amount,currency,notes,source_file,target_value,target_unit,programme_code
0,2026,Output,Kilometres of Roads graded,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,55,value,1
1,2026,Output,Number of Maternity Annexes constructed,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,1,count,1
2,2026,Output,Percentage of approved grant applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1
3,2026,Output,Percentage of approved loan applicants empowered,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1
4,2026,Output,Number of CDF meetings held,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,6,count,1
5,2026,Output,Number of Field appraisal reports produced,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,2,count,1
6,2026,Output,Number of Monitoring reports produced,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,5,count,1
7,2026,Output,Number of Desk Appraisal Reports produced,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,2,count,1
8,2026,Output,Percentage approved Skills bursaries applicant...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1
9,2026,Output,Percentage approved Secondary bursaries applic...,14,None,ZMW,Programme 1: Constituency Development; Key out...,chirundu_obb_2026.pdf.pdf,100,percent,1


In [20]:
expected_programmes = set(
    programmes_df["notes"].str.extract(
        r"Programme code (\d+)"
    )[0].dropna()
)

found_programmes = set(outputs_df["programme_code"].dropna())

counts = (
    outputs_df.groupby("programme_code")
    .size()
    .reset_index(name="output_count")
)

display(counts)

print(
    "Missing programmes:",
    sorted(expected_programmes - found_programmes, key=int)
)

,programme_code,output_count
0,1,10
1,10,5
2,11,6
3,12,1
4,13,12
5,15,4
6,18,18
7,2,8
8,3,21
9,4,2


Missing programmes: ['16', '17']


In [21]:
matches = raw_ocr[
    raw_ocr["ocr_text"].str.contains(
        r"Agricultural Services|Fisheries and Livestock",
        case=False,
        regex=True,
        na=False
    )
][["page", "ocr_text"]]

display(matches)

,page,ocr_text
6,7,OUTPUT BASED ANNUAL BUDGET Page7\n\nHEA 981 CH...
8,9,OUTPUT BASED ANNUAL BUDGET Page 9\n\nHEA 981 C...
10,11,OUTPUT BASED ANNUAL BUDGET Page 11\n\nHEA 981 ...
39,40,Page 40 OUTPUT BASED ANNUAL BUDGET\nHEA 981 CH...
40,41,OUTPUT BASED ANNUAL BUDGET Page 41\nHEA 981 CH...
42,43,OUTPUT BASED ANNUAL BUDGET Page 43\nHEA 981 CH...
43,44,Page 44 OUTPUT BASED ANNUAL BUDGET\nHEA 981 CH...
44,45,OUTPUT BASED ANNUAL BUDGET Page 45\nHEA 981 CH...
54,55,OUTPUT BASED ANNUAL BUDGET Page 55\nHEA 981 CH...


In [24]:
page_16_text = ocr_page(pdf_path, 42, psm=11)
page_17_text = ocr_page(pdf_path, 44, psm=11)

print("PROGRAMME 16 - PAGE 42")
print(page_16_text)

print("\n" + "=" * 80 + "\n")

print("PROGRAMME 17 - PAGE 44")
print(page_17_text)

PROGRAMME 16 - PAGE 42
Page 42

OUTPUT BASED ANNUAL BUDGET

HEA 981 CHIRUNDU TOWN COUNCIL

D

Agricultural enterprenuership trainings

01 Number of enterprenuership training conducted

02 Number of commodity market bulleting disseminated

36

52

Domestic Trade facilitated

01 Number of import and export permits facilitated

2,000

02 Number of trade data base established

03 Number of market linkages established

10

1,000

04 Number of inspections conducted

Market information on crop products and agriculture inputs

01 Number of Weekly Market repotrs

48

Market Research on crop products and inputs

01 Number of market research reports

02 Number of agricultural trade inspections conducted

300

500

Access to agriculuture credit improved

01 Number of farmers accessing agriculture credit

500

Administrative and support services provided

01 Number of times administrative and other support services provided

12

12

02 Number of vehicles insured,serviced and repaired

17

Monitorin

In [25]:
for page_number in [41, 42, 44, 45]:
    print("\n" + "=" * 80)
    print(f"PAGE {page_number}")
    print(ocr_page(pdf_path, page_number, psm=11))


PAGE 41
Page 41

OUTPUT BASED ANNUAL BUDGET

HEA 981 CHIRUNDU TOWN COUNCIL

D

Programme:

16 Agricultural Services

Table 6: Programme Outputs

Key Output and Output Indicator

2024

2025

2026

Target

Actual

Target

Actual*

Target

Crop Production Technologies disseminated

01 Number of technologies disseminated

15

15

Farmers trained in Climate Smart Agriculture

01 Number of farmers trained climate smart agriculture

6,500

6,500

02 Number of field officers trained in irrigation technologies

16

16

Demonstration plots established

01 Number of demonstration plots established

13

13

Farmer Field Schools established

01 Number of Farmer Field Schools established

26

26

02 Number of field officers trained in nutrition

16

16

Crop monitoring undertaken

01 Number of crop monitoring reports

24

24

Field days conducted

01 Number of field days conducted

02 Number of field officers trained in good farm management

16

16

Land under irrigation expanded

01 Number of irri

In [26]:
def add_missing_output(programme_code, programme_name, key_output,
                       indicator, page, target=None, unit="count"):

    exists = (
        outputs_df["notes"].str.contains(
            f"Programme {programme_code}:",
            regex=False,
            na=False
        )
        & outputs_df["programme_or_output"].eq(indicator)
    ).any()

    if not exists:
        outputs_df.loc[len(outputs_df)] = {
            "year": YEAR,
            "category": "Output",
            "programme_or_output": indicator,
            "page": page,
            "budget_amount": None,
            "currency": "ZMW",
            "notes": (
                f"Programme {programme_code}: {programme_name}; "
                f"Key output: {key_output}."
            ),
            "source_file": pdf_path.name,
            "target_value": target,
            "target_unit": unit,
            "programme_code": str(programme_code)
        }


# PROGRAMME 16: AGRICULTURAL SERVICES - pages 41 and 42
p16 = "Agricultural Services"

programme16_records = [
    ("Crop Production Technologies disseminated",
     "Number of technologies disseminated", 15),

    ("Farmers trained in Climate Smart Agriculture",
     "Number of farmers trained climate smart agriculture", 6500),

    ("Farmers trained in Climate Smart Agriculture",
     "Number of field officers trained in irrigation technologies", 16),

    ("Demonstration plots established",
     "Number of demonstration plots established", 13),

    ("Farmer Field Schools established",
     "Number of Farmer Field Schools established", 26),

    ("Farmer Field Schools established",
     "Number of field officers trained in nutrition", 16),

    ("Crop monitoring undertaken",
     "Number of crop monitoring reports", 24),

    ("Field days conducted",
     "Number of field days conducted", None),

    ("Field days conducted",
     "Number of field officers trained in good farm management", 16),

    ("Land under irrigation expanded",
     "Number of irrigation technologies promoted", None),

    ("Land under irrigation expanded",
     "Number of Farmers trained in irrigation", 500),

    ("Land under irrigation expanded",
     "Number of farmers adopting irrigation", 300),

    ("Land under irrigation expanded",
     "Hectares brought under irrigation", 200),

    ("Sustainable land management and land use planning promoted",
     "Number of farmers adopting sustainable practices", 200),

    ("Sustainable land management and land use planning promoted",
     "Hectares under sustainable agriculture", 50),

    ("Sustainable land management and land use planning promoted",
     "Number land use plan developed", None),

    ("Farm power and mechanization services promoted",
     "Number of mechanization service centres established", None),

    ("Farm power and mechanization services promoted",
     "Number of farmers trained in the use of mechanization", 1600),

    ("Farm power and mechanization services promoted",
     "Number of farmers accessing mechanisation services", 900),

    ("Nutrient dense crop production promoted",
     "Number of nutrition centres of excellence established", None),

    ("Processing and consumption of nutritious foods promoted",
     "Number of farmers trained in food and nutrition", 390),

    ("Agriculture information produced and disseminated",
     "Number of agriculture news and literature produced", 12),

    ("Agriculture information produced and disseminated",
     "Number of agriculture publications produced", 30),

    ("Agriculture shows organised and exhibited",
     "Number of agriculture shows organised and exhibited", None),

    ("Agricultural entrepreneurship trainings",
     "Number of entrepreneurship training conducted", None),

    ("Agricultural entrepreneurship trainings",
     "Number of commodity market bulletin disseminated", 52),

    ("Domestic Trade facilitated",
     "Number of import and export permits facilitated", 2000),

    ("Domestic Trade facilitated",
     "Number of trade data base established", None),

    ("Domestic Trade facilitated",
     "Number of market linkages established", 10),

    ("Domestic Trade facilitated",
     "Number of inspections conducted", 1000),

    ("Market information on crop products and agriculture inputs",
     "Number of Weekly Market reports", 48),

    ("Market Research on crop products and inputs",
     "Number of market research reports", None),

    ("Market Research on crop products and inputs",
     "Number of agricultural trade inspections conducted", 500),

    ("Access to agriculture credit improved",
     "Number of farmers accessing agriculture credit", 500),

    ("Administrative and support services provided",
     "Number of times administrative and other support services provided", 12),

    ("Administrative and support services provided",
     "Number of vehicles insured, serviced and repaired", 17),

    ("Monitoring and evaluation visits conducted",
     "Number of Monitoring and evaluation visits conducted", 12),

    ("Planning and Review meetings conducted",
     "Number of Departmental planning and review meetings held", 12),

    ("Planning and Review meetings conducted",
     "Number of Agricultural public functions attended", None),

    ("Planning and Review meetings conducted",
     "Number of quarterly and annual reports generated and submitted", None),

    ("Planning and review meetings conducted",
     "Number of Planning and review meetings held", 12),

    ("Planning and review meetings conducted",
     "Number of public events attended", None),

    ("Planning and review meetings conducted",
     "Number of quarterly and annual reports generated and submitted", None)
]

for key_output, indicator, target in programme16_records:
    add_missing_output(16, p16, key_output, indicator, 41, target)


# PROGRAMME 17: FISHERIES AND LIVESTOCK - pages 44 and 45
p17 = "Fisheries and Livestock"

programme17_records = [
    ("Participation in Fisheries and Livestock marketing improved",
     "Number of farmers trained in entrepreneurship and linked to the markets", 2500),

    ("Participation in Fisheries and Livestock marketing improved",
     "Number of farmers and traders issued with control of goods permits", 2000),

    ("Participation in Fisheries and Livestock marketing improved",
     "Number of Market days conducted", 12),

    ("Participation in Fisheries and Livestock marketing improved",
     "Number of trainings on nutrition conducted", 12),

    ("Participation in Fisheries and Livestock marketing improved",
     "Number of market price surveillance", 12),

    ("Extension visits conducted",
     "Number of extension visits conducted", 144),

    ("Livestock disease management conducted",
     "Number of farmers trained in Livestock disease management", 2500),

    ("Rabies vaccination conducted",
     "Number of Dogs vaccinated against rabies", 12000),

    ("Surveillance of diseases of national importance conducted",
     "Number of surveillances conducted", 240),

    ("Monitoring and evaluation",
     "Number of monitoring and supervision activities conducted", 60),

    ("Antimortem inspection",
     "Number of antimortem inspections conducted", 100),

    ("Regulation and enforcement",
     "Number of regulation and enforcement activities conducted", 360),

    ("Capture fisheries production and productivity improved",
     "Number of fishers trained in sustainable fishing and processing", 500),

    ("Capture fisheries production and productivity improved",
     "Number of patrols conducted", 72),

    ("Capture fisheries production and productivity improved",
     "Number of co-management structures created", 500),

    ("Capture fisheries production and productivity improved",
     "Number fishers registered and licensed", 500),

    ("Aquaculture extension and advisory services provided",
     "Number of fishers sensitised in good fishing methods", 500),

    ("Aquaculture extension and advisory services provided",
     "Number of farmers trained in Aquaculture practices", 600),

    ("Aquaculture extension and advisory services provided",
     "Number of fish farmer exchange visits conducted", None),

    ("Aquaculture extension and advisory services provided",
     "Number of dams stocked with fingerlings", 3),

    ("Livestock Production and Productivity improved",
     "Number of livestock farmers receiving extension services", 10000),

    ("Livestock Production and Productivity improved",
     "Number of livestock field days conducted", 12),

    ("Livestock Production and Productivity improved",
     "Number of farmers to receive climate smart livestock technologies and practices", 24),

    ("Livestock Production and Productivity improved",
     "Number of exchange visits conducted", None),

    ("Livestock Production and Productivity improved",
     "Number of farm visits to livestock farmers conducted", None),

    ("Livestock Production and Productivity improved",
     "Number of rangeland mapped", 3),

    ("Livestock Production and Productivity improved",
     "Smallholder farmers supported in pasture seed", 120),

    ("Pasture and rangeland management improved",
     "Number of Smallholder farmers supported in pasture seed", 80),

    ("Pasture and rangeland management improved",
     "Number of Farmers trained in utilization of pasture", 1000),

    ("Pasture and rangeland management improved",
     "Number of stakeholder engagements in feed additives", 3),

    ("Pasture and rangeland management improved",
     "Number of rangeland mapped", 3),

    ("Pasture and rangeland management improved",
     "Number of rangeland committees formed", 3),

    ("Pasture and rangeland management improved",
     "Number of hectares brought under rangeland", 150000),

    ("Pasture and rangeland management improved",
     "Number of rangeland rehabilitated", 1),

    ("Back stopping and supervision of camps conducted",
     "Number of backstopping, supervision activities conducted", 48),

    ("Back stopping and supervision of camps conducted",
     "Number of Government functions, events and meetings attended", 60),

    ("Management and support services provided",
     "Number of administration support provided", 12),

    ("Management and support services provided",
     "Number of staff returns and Financial reports done", 12),

    ("Management and support services provided",
     "Number of shows attended", 12),

    ("Management and support services provided",
     "Number of personal emoluments paid", None),

    ("Management and support services provided",
     "Number of Utility bills and rentals paid", 12)
]

for key_output, indicator, target in programme17_records:
    add_missing_output(17, p17, key_output, indicator, 44, target)

print("Programme 16 rows:", (outputs_df["programme_code"] == "16").sum())
print("Programme 17 rows:", (outputs_df["programme_code"] == "17").sum())
print("Total output rows:", len(outputs_df))

Programme 16 rows: 42
Programme 17 rows: 40
Total output rows: 204


In [27]:
# Remove helper column before combining datasets.
outputs_for_export = outputs_df[extracted_columns].copy()

final_df = pd.concat(
    [
        financial_df[extracted_columns],
        programmes_df[extracted_columns],
        outputs_for_export
    ],
    ignore_index=True
)

final_df["budget_amount"] = pd.to_numeric(
    final_df["budget_amount"],
    errors="coerce"
).astype("Int64")

final_df["target_value"] = pd.to_numeric(
    final_df["target_value"],
    errors="coerce"
).astype("Int64")

final_file = output_folder / (
    "db-unza26-csc4792-"
    "chirundu_town_council_2026_extracted.csv"
)

final_df.to_csv(
    final_file,
    sep="|",
    index=False,
    encoding="utf-8-sig"
)

print(final_df["category"].value_counts())
print("Total final rows:", len(final_df))
print("Created:", final_file)

category
Output           204
Local Revenue     49
Programme         16
Revenue            5
CDF                1
LGEF               1
Name: count, dtype: int64
Total final rows: 276
Created: output_2026\db-unza26-csc4792-chirundu_town_council_2026_extracted.csv


In [28]:
check_df = pd.read_csv(final_file, sep="|")

print("Rows read back:", len(check_df))
print("Columns:", check_df.columns.tolist())

assert len(check_df) == len(final_df)
assert "category" in check_df.columns
assert "budget_amount" in check_df.columns
assert "target_value" in check_df.columns

print("2026 CSV validation passed.")

Rows read back: 276
Columns: ['year', 'category', 'programme_or_output', 'page', 'budget_amount', 'currency', 'notes', 'source_file', 'target_value', 'target_unit']
2026 CSV validation passed.


In [29]:
missing_targets = final_df[
    (final_df["category"] == "Output")
    & (final_df["target_value"].isna())
]

display(
    missing_targets[
        ["page", "programme_or_output", "notes", "target_unit"]
    ]
)

print("Outputs with missing targets:", len(missing_targets))

,page,programme_or_output,notes,target_unit
201,41,Number of field days conducted,Programme 16: Agricultural Services; Key outpu...,count
203,41,Number of irrigation technologies promoted,Programme 16: Agricultural Services; Key outpu...,count
209,41,Number land use plan developed,Programme 16: Agricultural Services; Key outpu...,count
210,41,Number of mechanization service centres establ...,Programme 16: Agricultural Services; Key outpu...,count
213,41,Number of nutrition centres of excellence esta...,Programme 16: Agricultural Services; Key outpu...,count
217,41,Number of agriculture shows organised and exhi...,Programme 16: Agricultural Services; Key outpu...,count
218,41,Number of entrepreneurship training conducted,Programme 16: Agricultural Services; Key outpu...,count
221,41,Number of trade data base established,Programme 16: Agricultural Services; Key outpu...,count
225,41,Number of market research reports,Programme 16: Agricultural Services; Key outpu...,count
232,41,Number of Agricultural public functions attended,Programme 16: Agricultural Services; Key outpu...,count


Outputs with missing targets: 16
